In [ ]:
from textacy import preprocessing
import trafilatura
from bs4 import BeautifulSoup as bs

In [ ]:
url = "https://www.weroad.it/viaggi"
downloaded = trafilatura.fetch_url(url)
text = trafilatura.extract(downloaded)
text = preprocessing.normalize.whitespace(text)

soup = bs(downloaded, 'lxml')
title = soup.select_one('title').text


In [ ]:
for link in soup.find_all('a'):
    print(link.get('href'))


In [ ]:
tipo_viaggi = ["/viaggi-avventura-weroad",
"/viaggi-beach-life",
"/viaggi-barca-a-vela",
"/weekend-organizzati",
"/weroad-collection",
"/viaggi-weroadx",
"/viaggi-5-giorni",
"/viaggi-zaino-in-spalla",
"/viaggi-trekking",
"/viaggi-safari",
"/viaggi-aurora-boreale",
"/settimana-bianca"]


In [ ]:
viaggi_continenti = ["/viaggi-asia",
"/viaggi-sud-est-asiatico",
"/viaggi-middle-east",
"/viaggi-itinerari-nord-europa",
"/europaovest",
"/viaggi-francia",
"/viaggi-itinerari-italia",
"/viaggi-itinerari-grecia",
"/viaggi-est-europa",                     
"/viaggi-africa",
"/viaggi-nord-america",
"/viaggi-centro-america",
"/viaggi-sud-america",
"/viaggi-oceania"]


In [ ]:
import pandas as pd
df = pd.DataFrame(columns = ["Country", "Nome_Viaggio", "Link", "Descrizione Viaggio", "Mood Viaggio", "Itinerario"])
for c in viaggi_continenti:
    url = f"https://www.weroad.it{c}"
    downloaded = trafilatura.fetch_url(url)
    soup = bs(downloaded, 'lxml')
    list_soups_countries = soup.find_all(attrs={'class':'wr-travels wr-scroll-product'})
    for soup_country in list_soups_countries:
        country_name = soup_country.find(attrs={'class':"title m-auto mb-0 block font-display leading-tight text-3xl lg:text-4xl"}).text
        print(country_name)
        for link in soup_country.find_all('a'):
            l = link.find(attrs={'class':'lg:hover:underline text-gray-700 lg:hover:text-gray-900'})
            if l:
                nome_viaggio = l.text
                print(nome_viaggio)
                
                link_viaggio = link.get('href')
                url_viaggio = f"https://www.weroad.it{link_viaggio}"
                url_itinerario = f"https://www.weroad.it{link_viaggio}/itinerario"
                
                downloaded_viaggio = trafilatura.fetch_url(url_viaggio)
                soup_viaggio = bs(downloaded_viaggio, 'lxml')
                
                mood_viaggio = {}
                moods = soup_viaggio.find_all(attrs={'class':'px-[15px] lg:w-1/5'})
                for mood in moods:
                    info = mood.find(attrs = {'class': 'text'})
                    tipo_mood = info.find('div', attrs={'data-v-2a2f4642': ''}).text
                    percentage_mood = info.find('strong', attrs={'data-v-2a2f4642': ''}).text
                    mood_viaggio[tipo_mood] = percentage_mood
                
                text_description = soup_viaggio.find(attrs={'class': 'long-description to-animate-modal text-base leading-relaxed'}).text.split("\n")
                text_description = ['Il viaggio in pillole'] + text_description
                descrizione_viaggio = {}
                for i in range(0, len(text_description), 2):
                    subtitle = text_description[i]
                    desc = text_description[i+1]
                    descrizione_viaggio[subtitle] = desc
                
                downloaded_itinerario = trafilatura.fetch_url(url_itinerario)
                soup_itinerario = bs(downloaded_itinerario, 'lxml')
                itinerario = {}
                for day, it in enumerate(soup_itinerario.find_all(attrs={"class": 'itinerary-item'})):
                    itinerario[f"Giorno {day+1}"] = {}
                    it_title = it.find(attrs={"class": "text-lg font-bold leading-tight lg:text-2xl"}).text.strip()
                    itinerario[f"Giorno {day+1}"]['Title'] = it_title
                    tappe_nel_giorno = []
                    it_descr_list = it.find_all(attrs={"class": "sub-stage"})
                    for desc in it_descr_list:
                        header = desc.find(attrs={'class': "header-substage mb-[5px]"})
                        if header:
                            header = header.text
                        else:
                            continue
                        desc_text = desc.find(attrs={'class': "description content"})
                        if desc_text:
                            desc_text = desc_text.text
                        else:
                            desc_text = ""
                        tappe_nel_giorno.append({"Subtitle": header,
                                                "Text": desc_text})
                    itinerario[f"Giorno {day+1}"]['Tappe'] = tappe_nel_giorno
                    day_activities = it.find(attrs={"class": "moods flex mt-[20px]"})
                    act_type = {}
                    for c in day_activities.contents:
                        activity_type = c.text.strip()
                        percentage = c.get('data-title').strip()
                        act_type[activity_type] = percentage
                    itinerario[f"Giorno {day+1}"]['Activity_Types'] = act_type

                row = pd.DataFrame.from_dict({"Country": [country_name],
                                             "Nome_Viaggio": [nome_viaggio],
                                              "Link": [link_viaggio],
                                              "Descrizione Viaggio": [descrizione_viaggio],
                                              "Mood Viaggio": [mood_viaggio],
                                             "Itinerario": [itinerario]})
                df = pd.concat([df, row], ignore_index=True)
                print("-"*80)
        print("*"*80)

In [ ]:
df

In [ ]:
df[df['Nome_Viaggio'].str.lower().str.contains("madeira")]

In [ ]:
for i,row in df[df['Nome_Viaggio'].str.lower().str.contains("madeira")].iterrows():
    print(row['Nome_Viaggio'])
    print(row['Descrizione Viaggio'])
    print(row['Itinerario'])
    print("*"*80)

In [ ]:
df.to_pickle("df_viaggi_weroad.pkl")

In [ ]:
import pandas as pd
df = pd.read_pickle("df_viaggi_weroad.pkl")

In [ ]:
df[df['Nome_Viaggio'].str.lower().str.contains("madeira")]

In [ ]:
df['Country'].tolist()

In [ ]:
df.iloc[8]['Itinerario']

In [ ]:
for i, row in df.iterrows():
    if row['Mood Viaggio']=={}:
        print(i)
    print(row['Mood Viaggio'])

In [ ]:
url_viaggio

In [ ]:
url = 'https://www.weroad.it/viaggi/giappone-360-shinkansen-experience-weroadx'
downloaded = trafilatura.fetch_url(url)
soup = bs(downloaded, 'lxml')

text = trafilatura.extract(downloaded)
text = preprocessing.normalize.whitespace(text)


In [ ]:
text

In [ ]:
soup.find(attrs={'class': 'long-description to-animate-modal text-base leading-relaxed'}).text.split("\n")

In [ ]:
url = f"https://www.weroad.it/viaggi/giappone/itinerario"
downloaded = trafilatura.fetch_url(url)
soup = bs(downloaded, 'lxml')

text = trafilatura.extract(downloaded)
text = preprocessing.normalize.whitespace(text)

In [ ]:
text.split("\n")

In [ ]:
day_activities = it.find(attrs={"class": "moods flex mt-[20px]"})
for c in day_activities.contents:
    activity_type = c.text.strip()
    percentage = c.get('data-title').strip()

In [ ]:
for day, it in enumerate(soup.find_all(attrs={"class": 'itinerary-item'})):
    print(f"Giorno {day+1}")
    it_title = it.find(attrs={"class": "text-lg font-bold leading-tight lg:text-2xl"}).text.strip()
    print(it_title)
    it_descr_list = it.find_all(attrs={"class": "sub-stage"})
    for desc in it_descr_list:
        header = desc.find(attrs={'class': "header-substage mb-[5px]"}).text
        desc_text = desc.find(attrs={'class': "description content"}).text
        print(header)
        print(desc_text)
    day_activities = it.find(attrs={"class": "moods flex mt-[20px]"})
    for c in day_activities.contents:
        activity_type = c.text.strip()
        percentage = c.get('data-title').strip()
        print(f"{activity_type} - {percentage}")
    print("-"*80)


In [ ]:
<div data-v-49edf6bd="" class="moods flex mt-[20px]" bis_skin_checked="1"><!----><!----><!----><div data-v-49edf6bd="" data-title="culture" class="mood" bis_skin_checked="1"><div data-v-49edf6bd="" class="icon" bis_skin_checked="1"><img data-v-49edf6bd="" decoding="async" alt="culture" title="culture" class="block" data-src="https://cdn.weroad.io/common/images/moods/culture-black.svg" src="https://cdn.weroad.io/common/images/moods/culture-black.svg" lazy="loaded"></div> <div data-v-49edf6bd="" class="value lg:text-md mt-[5px] whitespace-nowrap text-center text-sm font-bold text-culture" bis_skin_checked="1">
                  100 %
                </div></div><!----></div>

In [ ]:
it_descr

In [ ]:
soup

In [ ]:
soup.get_text()

In [ ]:
for link in soup.body.find_all('a'):
    print(link.get('href'))


In [ ]:
for i, c in enumerate(soup.body.contents):
    print(i)
    try:
        for link in c.find_all('a'):
            print(link.get('href'))
    except:
        print("no link")
    print("-"*80)
    

In [ ]:
soup.body.contents[2]

In [ ]:
soup.find("wr-travels-container light travel-default")

In [ ]:
ss = soup.find_all(attrs={'class':'wr-travels wr-scroll-product'})

In [ ]:
ss[0]

In [ ]:
for link in ss.find_all('a'):
    print(link.get('href'))
